# DVA Lab#1
**Dataset:** UCI Adult dataset
**Roll Number_Section:** 24i-2559_DSA

In [ ]:
import numpy as np
import pandas as pd
import sqlite3

pd.set_option('display.max_columns', None)

In [18]:

columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]
df = pd.read_csv('adult.csv', header=0, names=columns, skipinitialspace=True)
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


## Task 1

In [19]:
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education-num   32561 non-null  int64 
 5   marital-status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital-gain    32561 non-null  int64 
 11  capital-loss    32561 non-null  int64 
 12  hours-per-week  32561 non-null  int64 
 13  native-country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [20]:

print("Missing values BEFORE fix:")
print(df.isnull().sum())

Missing values BEFORE fix:
age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64


In [21]:
df.replace(' ?', np.nan, inplace=True)
df.replace('?', np.nan, inplace=True)  

print("Missing values AFTER fix:")
print(df.isnull().sum())

Missing values AFTER fix:
age                  0
workclass         1836
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     583
income               0
dtype: int64


From the missing value report, the columns with the most missing data are occupation (1843, 5.66%), workclass (1836, 5.64%) and native-country (583, 1.79%). These are the same columns mentioned in the lab manual, so it matches what was expected.

## Task 2

In [22]:
missing_count = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)

missing_report = pd.DataFrame({'missing_count': missing_count, 'missing_pct': missing_pct})
missing_report = missing_report[missing_report['missing_count'] > 0].sort_values('missing_count', ascending=False)
missing_report

,missing_count,missing_pct
occupation,1843,5.66
workclass,1836,5.64
native-country,583,1.79


**Observation:** workclass, occupation, and native-country have the most missing data,
as expected,  these are self-reported categorical fields prone to non-response.

## Task 3

In [23]:

df['workclass'] = df['workclass'].fillna('Unknown')
df['occupation'] = df['occupation'].fillna('Unknown')


df['native-country'] = df['native-country'].fillna(df['native-country'].mode()[0])


numeric_cols = df.select_dtypes(include=[np.number]).columns
for c in numeric_cols:
    if df[c].isnull().sum() > 0:
        df[c] = df[c].fillna(df[c].median())

print("Missing values after handling:")
print(df.isnull().sum().sum(), "total missing values remain")

Missing values after handling:
0 total missing values remain


For workclass and occupation, I filled the missing values with "Unknown" instead of dropping the rows. I did this because dropping would remove around 5-6% of the data which is too much, and "Unknown" still keeps the row useful for other columns.

For native-country, I filled missing values with the mode (United-States) because only 1.79% was missing and most people already belong to that category, so it won't really change the distribution.

There were no missing numeric columns left after fixing the hidden "?" values, so no numeric imputation was needed.

## Task 4 

In [24]:

exact_dupes = df.duplicated().sum()
print("Exact duplicate rows:", exact_dupes)

cols_no_label = [c for c in df.columns if c != 'income']
dupes_no_label = df.duplicated(subset=cols_no_label).sum()
print("Duplicates ignoring 'income' label:", dupes_no_label)

df = df.drop_duplicates()
print("Shape after dropping exact duplicates:", df.shape)

Exact duplicate rows: 24
Duplicates ignoring 'income' label: 25
Shape after dropping exact duplicates: (32537, 15)


There were 24 exact duplicate rows which I removed. When checking duplicates while ignoring the income column, there were 25 (one extra). I decided not to remove that extra row because it could be two different people who just happen to have the same attributes but different income, so it's not a real duplicate.

## Task 5 

In [25]:
for col in ['education', 'marital-status', 'native-country']:
    print(f"--- {col} ---")
    print(df[col].unique())
    print()

--- education ---
['HS-grad' 'Some-college' '7th-8th' '10th' 'Doctorate' 'Prof-school'
 'Bachelors' 'Masters' '11th' 'Assoc-acdm' 'Assoc-voc' '1st-4th' '5th-6th'
 '12th' '9th' 'Preschool']

--- marital-status ---
['Widowed' 'Divorced' 'Separated' 'Never-married' 'Married-civ-spouse'
 'Married-spouse-absent' 'Married-AF-spouse']

--- native-country ---
['United-States' 'Mexico' 'Greece' 'Vietnam' 'China' 'Taiwan' 'India'
 'Philippines' 'Trinadad&Tobago' 'Canada' 'South' 'Holand-Netherlands'
 'Puerto-Rico' 'Poland' 'Iran' 'England' 'Germany' 'Italy' 'Japan' 'Hong'
 'Honduras' 'Cuba' 'Ireland' 'Cambodia' 'Peru' 'Nicaragua'
 'Dominican-Republic' 'Haiti' 'El-Salvador' 'Hungary' 'Columbia'
 'Guatemala' 'Jamaica' 'Ecuador' 'France' 'Yugoslavia' 'Scotland'
 'Portugal' 'Laos' 'Thailand' 'Outlying-US(Guam-USVI-etc)']



In [26]:

obj_cols = df.select_dtypes(include='object').columns
for c in obj_cols:
    df[c] = df[c].str.strip()

for col in ['education', 'marital-status', 'native-country']:
    print(f"--- {col} (cleaned) ---")
    print(df[col].unique())
    print()

--- education (cleaned) ---
['HS-grad' 'Some-college' '7th-8th' '10th' 'Doctorate' 'Prof-school'
 'Bachelors' 'Masters' '11th' 'Assoc-acdm' 'Assoc-voc' '1st-4th' '5th-6th'
 '12th' '9th' 'Preschool']

--- marital-status (cleaned) ---
['Widowed' 'Divorced' 'Separated' 'Never-married' 'Married-civ-spouse'
 'Married-spouse-absent' 'Married-AF-spouse']

--- native-country (cleaned) ---
['United-States' 'Mexico' 'Greece' 'Vietnam' 'China' 'Taiwan' 'India'
 'Philippines' 'Trinadad&Tobago' 'Canada' 'South' 'Holand-Netherlands'
 'Puerto-Rico' 'Poland' 'Iran' 'England' 'Germany' 'Italy' 'Japan' 'Hong'
 'Honduras' 'Cuba' 'Ireland' 'Cambodia' 'Peru' 'Nicaragua'
 'Dominican-Republic' 'Haiti' 'El-Salvador' 'Hungary' 'Columbia'
 'Guatemala' 'Jamaica' 'Ecuador' 'France' 'Yugoslavia' 'Scotland'
 'Portugal' 'Laos' 'Thailand' 'Outlying-US(Guam-USVI-etc)']



## Task 6 

In [27]:
df.describe()

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,32537.000000,3.253700e+04,32537.000000,32537.000000,32537.000000,32537.000000
mean,38.585549,1.897808e+05,10.081815,1078.443741,87.368227,40.440329
std,13.637984,1.055565e+05,2.571633,7387.957424,403.101833,12.346889
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.369930e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [28]:
df.describe(include='object')

,workclass,education,marital-status,occupation,relationship,race,sex,native-country,income
count,32537,32537,32537,32537,32537,32537,32537,32537,32537
unique,9,16,7,15,6,5,2,41,2
top,Private,HS-grad,Married-civ-spouse,Prof-specialty,Husband,White,Male,United-States,<=50K
freq,22673,10494,14970,4136,13187,27795,21775,29735,24698


In [29]:
mean_age, median_age = df['age'].mean(), df['age'].median()
mean_hours, median_hours = df['hours-per-week'].mean(), df['hours-per-week'].median()

print(f"Age       -> mean: {mean_age:.2f}, median: {median_age:.2f}")
print(f"Hours/wk  -> mean: {mean_hours:.2f}, median: {median_hours:.2f}")

Age       -> mean: 38.59, median: 37.00
Hours/wk  -> mean: 40.44, median: 40.00


Mean age is 38.59 and median age is 37, which are pretty close, so the age distribution is not badly skewed. Mean hours-per-week is 40.44 and median is 40, also very close, which makes sense since most people work a normal 40 hour week. So overall, no big difference between mean and median for either column.

## Task 7 

In [30]:
top_occupation = df['occupation'].value_counts().idxmax()
print("Most common occupation:", top_occupation)
df['occupation'].value_counts().head()

Most common occupation: Prof-specialty


occupation
Prof-specialty     4136
Craft-repair       4094
Exec-managerial    4065
Adm-clerical       3768
Sales              3650
Name: count, dtype: int64

In [31]:
sex_pct = (df['sex'].value_counts(normalize=True) * 100).round(2)
sex_pct

sex
Male      66.92
Female    33.08
Name: proportion, dtype: float64

In [32]:
income_pct = (df['income'].value_counts(normalize=True) * 100).round(2)
income_pct

income
<=50K    75.91
>50K     24.09
Name: proportion, dtype: float64

**Comment:** The income classes are clearly **imbalanced**, the <=50K class have
the large majority (roughly 3-to-1) compared to >50K, which is important to keep in mind
for any downstream classification task.

## Task 8 

In [33]:
edu_mapping = df.groupby('education')['education-num'].unique()
edu_mapping

education
10th             [6]
11th             [7]
12th             [8]
1st-4th          [2]
5th-6th          [3]
7th-8th          [4]
9th              [5]
Assoc-acdm      [12]
Assoc-voc       [11]
Bachelors       [13]
Doctorate       [16]
HS-grad          [9]
Masters         [14]
Preschool        [1]
Prof-school     [15]
Some-college    [10]
Name: education-num, dtype: object

In [ ]:
inconsistent = edu_mapping[edu_mapping.apply(len) > 1]
if inconsistent.empty:
    print("No inconsistencies found: each education level maps to exactly one education-num.")
else:
    print("Inconsistent mappings found:")
    print(inconsistent)

No inconsistencies found: each education level maps to exactly one education-num.


After checking education against education-num, every education level maps to exactly one number with no mismatches. So the data is consistent here, no inconsistencies found.

## Task 9

In [ ]:

conn = sqlite3.connect('adult_income.db')
df.to_sql('adult_income', conn, if_exists='replace', index=False)


query = "SELECT * FROM adult_income WHERE age > 30;"
df_sql = pd.read_sql_query(query, conn)

print("Original cleaned DataFrame shape:", df.shape)
print("SQL query result shape (age > 30):", df_sql.shape)

df_pandas_check = df[df['age'] > 30]
print("Pandas-filtered shape (age > 30):", df_pandas_check.shape)
print("Shapes match:", df_sql.shape == df_pandas_check.shape)

conn.close()

Original cleaned DataFrame shape: (32537, 15)
SQL query result shape (age > 30): (21979, 15)
Pandas-filtered shape (age > 30): (21979, 15)
Shapes match: True


The SQL query (age > 30) returned 21979 rows, and when I filtered the same condition using pandas directly, it also gave 21979 rows. Both shapes match, so this confirms the SQL extraction was done correctly.

## Task 10 - Report

**Dataset Overview:**
The dataset I used is the UCI Adult dataset. After cleaning it, there are 32537 rows and 15 columns left (started with 32561, removed some duplicates). Each row is basically one person's info like their age, job, education, work hours etc, and the goal is to predict if that person earns more than 50K a year or not.

**Data Quality Issues Found:**
- The dataset had missing values but they were hidden as "?" instead of actual NaN, so at first isnull() was showing 0 missing values which was wrong. After replacing "?" with NaN, I found:
  - occupation had 1843 missing values (5.66%)
  - workclass had 1836 missing values (5.64%)
  - native-country had 583 missing values (1.79%)
- There were 24 exact duplicate rows which I dropped. When I checked duplicates without looking at the income column, there were 25, so I kept the extra one since two different people can have same info but different income.
- Checked education, marital-status and native-country for whitespace issues but didn't really find any inconsistency after stripping, values were already fine.

**Key Observations:**
- Income column is imbalanced, about 76% people have <=50K and only 24% have >50K.
- 67% of the people in the dataset are Male and 33% are Female.
- Most common occupation is Prof-specialty.
- Mean age is 38.59 and median is 37, so not a big difference, distribution looks fairly normal.
- Mean hours-per-week is 40.44 and median is 40, which makes sense since most people work regular 40 hour week jobs.
- Checked education vs education-num and both matched perfectly for every row, no mismatch found.

**Cleaning Decisions Made:**
- Replaced "?" with NaN so I could actually see the missing values properly.
- For workclass and occupation I filled missing values with "Unknown" instead of dropping rows because dropping would lose too much data (~5% each).
- For native-country I filled missing values with the mode (United-States) since only 1.79% was missing and most people are already from United-States anyway.
- Removed the 24 exact duplicate rows but kept rows that only differed in the income column since those are different people.
- Also did a small SQL check (age > 30) and compared the result with pandas, shapes matched so the data extraction was correct.

In [37]:
# Save cleaned dataset to CSV
df.to_csv('cleaned_adult.csv', index=False)